# Setup — Multimedia Retrieval HS26

Run this notebook once after cloning the repository, and again whenever a new chapter adds
dependencies. It fetches everything `uv sync` cannot install on its own: NLTK corpora, spaCy
models, and the lecture PDFs and datasets behind the document collections.

### Software you need

**[uv](https://docs.astral.sh/uv/)** is the only tool you have to install yourself. It manages
both the virtual environment and the Python interpreter, so a separate Python installation is
not required — uv downloads Python 3.12 for this project automatically.

**Windows** — in PowerShell:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

**macOS** — in Terminal:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Package managers work as well: `brew install uv` on macOS,
`winget install --id=astral-sh.uv -e` on Windows. Afterwards open a **new** terminal — the
installer changes your `PATH` — and confirm with `uv --version`.

If you would rather manage Python yourself, install 3.12 or newer from
[python.org/downloads](https://www.python.org/downloads/), or let uv do it with
`uv python install 3.12`.

### Create the environment

From the repository root:

```bash
uv sync
```

Then select `.venv` as the kernel for this notebook — in VS Code: *Select Kernel* →
*Python Environments* → `.venv` — and run all cells. Every cell is idempotent, so re-running
the notebook only fetches what is missing.

## 1. Check the environment

Confirms that the notebook runs inside the project's `.venv` and that the packages the
notebooks import are importable.

In [ ]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

print(f"Python      {sys.version.split()[0]}")
print(f"Interpreter {sys.executable}")
print(f"In .venv    {'.venv' in sys.executable}")
print()

# Packages imported by the demo and exercise notebooks
PACKAGES = [
    "matplotlib",
    "numpy",
    "ipywidgets",
    "nltk",
    "PyPDF2",
    "tabulate",
    "spacy",
    "kagglehub",
    "sklearn",
    "unidecode",
    "wordfreq",
    "bs4",
    "langchain_core",
]

missing = []
for name in PACKAGES:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "?")
        print(f"  OK      {name} {version}")
    except ImportError:
        missing.append(name)
        print(f"  MISSING {name}")

if missing:
    print(f"\nRun `uv sync` (or `uv add {' '.join(missing)}`) and restart the kernel.")


## 2. NLTK corpora

`pip install nltk` ships the code but no data. The demos call `shared.text.stopwords_for()`,
which reads the **stopwords** corpus; the later text-processing chapters add sentence
segmentation, lemmatization, and POS tagging.

Downloads land in `~/nltk_data` and are shared across projects.

In [ ]:
import nltk

# (package id, path used by nltk.data.find, what it is for)
NLTK_RESOURCES = [
    ("stopwords", "corpora/stopwords", "stop word lists (ch01 onwards)"),
    ("punkt", "tokenizers/punkt", "sentence segmentation"),
    ("punkt_tab", "tokenizers/punkt_tab", "sentence segmentation tables (NLTK >= 3.8.2)"),
    ("wordnet", "corpora/wordnet", "lemmatization + synonyms"),
    ("omw-1.4", "corpora/omw-1.4", "multilingual WordNet"),
    ("averaged_perceptron_tagger_eng", "taggers/averaged_perceptron_tagger_eng", "POS tagging"),
]


def is_cached(resource_path):
    """True if the resource is available, unpacked or still zipped."""
    for candidate in (resource_path, f"{resource_path}.zip"):
        try:
            nltk.data.find(candidate)
            return True
        except LookupError:
            continue
    return False


for package_id, resource_path, purpose in NLTK_RESOURCES:
    if is_cached(resource_path):
        print(f"  cached      {package_id:<32} {purpose}")
        continue
    print(f"  downloading {package_id:<32} {purpose}")
    nltk.download(package_id, quiet=True)
    if not is_cached(resource_path):
        print(f"  FAILED      {package_id} — check your network, then re-run this cell")

In [ ]:
# Verify: stopwords in the languages the collections use, plus stemming and lemmatization
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

for language in ("english", "german", "french"):
    words = stopwords.words(language)
    print(f"  {language:<8} {len(words):>3} stop words — e.g. {', '.join(words[:6])}")

sample = ["retrieval", "ranking", "documents", "better"]
print("\n  Porter stemmer:  ", [PorterStemmer().stem(w) for w in sample])
print("  WordNet lemmas:  ", [WordNetLemmatizer().lemmatize(w) for w in sample])

## 3. spaCy models

spaCy needs a trained pipeline for tokenization, POS tagging, lemmatization, and NER.
`en_core_web_sm`, `de_core_news_sm`, and `fr_core_news_sm` are declared as project
dependencies (see `[tool.uv.sources]` in `pyproject.toml`), so `uv sync` installs all
three and the check below should just pass.

If one is missing, the cell installs it. Note that `python -m spacy download` does **not**
work here: uv-managed virtual environments have no `pip`, so we install the model wheel
with `uv pip install` instead.

In [ ]:
import subprocess

import spacy

SPACY_MODELS = ["en_core_web_sm", "de_core_news_sm", "fr_core_news_sm"]

for SPACY_MODEL in SPACY_MODELS:
    MODEL_WHEEL = (
        "https://github.com/explosion/spacy-models/releases/download/"
        f"{SPACY_MODEL}-3.8.0/{SPACY_MODEL}-3.8.0-py3-none-any.whl"
    )

    if spacy.util.is_package(SPACY_MODEL):
        print(f"  cached      {SPACY_MODEL}")
    else:
        print(f"  downloading {SPACY_MODEL}")
        result = subprocess.run(["uv", "pip", "install", MODEL_WHEEL], capture_output=True, text=True)
        print(result.stdout or result.stderr)
        if result.returncode != 0:
            print("Install it from a terminal instead:")
            print(f'  uv pip install "{MODEL_WHEEL}"')

nlp = spacy.load("en_core_web_sm")
doc = nlp("Multimedia retrieval ranks documents at the University of Basel.")

print("\n  token        pos     lemma")
for token in doc:
    print(f"  {token.text:<12} {token.pos_:<7} {token.lemma_}")
print("\n  entities:", [(ent.text, ent.label_) for ent in doc.ents])

## 4. Collection data

The collections fetch their source data on first use and cache it under
`demos/data/.cache/` and `exercises/data/.cache/` — both gitignored, and both folders keep
their own copy of `shared/`.

First the lecture PDFs from the DMI web server (~30 MB per folder), then the Kaggle movie
dataset used by the ch01 exercise (~250 MB, downloaded once into `~/.cache/kagglehub` and
shared between the two folders). Warming the caches now means the notebooks run offline
afterwards — skip the next two cells if you are on a metered connection.

In [ ]:
import urllib.request


def with_shared(folder):
    """Make `shared` importable from demos/ or exercises/ — each folder has its own copy."""
    path = str(PROJECT_ROOT / folder)
    for module in [m for m in list(sys.modules) if m.startswith("shared")]:
        del sys.modules[module]
    sys.path.insert(0, path)
    return path


for folder in ("demos", "exercises"):
    path = with_shared(folder)
    try:
        from shared.collections import CACHE_DIR
        from shared.collections.slides import SLIDE_CATALOGUE

        print(f"{folder}/ → {CACHE_DIR.relative_to(PROJECT_ROOT)}")
        for name, info in SLIDE_CATALOGUE.items():
            target = CACHE_DIR / "slides" / f"{name}.pdf"
            if target.exists():
                print(f"  cached      {name}")
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            try:
                urllib.request.urlretrieve(info["url"], target)
                print(f"  downloading {name} ({target.stat().st_size / 1e6:.1f} MB)")
            except Exception as error:  # lecture not published yet, or no network
                target.unlink(missing_ok=True)
                print(f"  skipped     {name} — {type(error).__name__}: {error}")
    finally:
        sys.path.remove(path)

In [ ]:
# The ch01 exercise opens with load_collection("movies-small"), which pulls the Kaggle
# dataset via kagglehub. The dataset is public, so no Kaggle token is needed.
for folder in ("demos", "exercises"):
    path = with_shared(folder)
    try:
        from shared.collections import load_collection

        print(f"{folder}/")
        movies = load_collection("movies-small")
        print(f"  {len(movies)} movies cached — e.g. {movies.documents()[0]['title']}")
    except Exception as error:
        print(f"  skipped — {type(error).__name__}: {error}")
    finally:
        sys.path.remove(path)

## 5. Optional extras

Not needed for the chapters published so far. Set them up when a notebook asks for them.

| Extra | Needed for | How |
|---|---|---|
| `boto3` + AWS credentials | the LLM / RAG demos in `shared/llm.py` | `uv add boto3`, then `aws login` (Bedrock access in `eu-central-1`) |
| `matplotlib-venn` | `shared.plot.plot_venn_diagram` | `uv add matplotlib-venn` |

Packages such as **scipy** and **pandas** need no extra download step —
`uv add scipy` is enough. Only NLTK and spaCy ship their data separately from the package,
which is why they get their own sections above.

## 6. Smoke test

Loads a real collection through the same code path as the notebooks. If this prints a
document count and a few terms, you are ready to open `demos/ch01-00-explore-collection.ipynb`.

In [ ]:
with_shared("demos")

from shared.collections import load_collection
from shared.text import pipeline

collection = load_collection("slides-classical-text-retrieval")
terms = pipeline(collection.documents()[0]["text"])

print(f"  {collection.name}: {len(collection)} documents")
print(f"  first page, processed: {terms[:12]}")
print("\nSetup complete — open demos/ch01-00-explore-collection.ipynb next.")